# Perhitungan Seluruh Fitur & Metrik Turunan Dashboard Sentimen M-Banking

Notebook ini berisi seluruh fungsi perhitungan agregasi, metrik turunan, KPI *scorecards*, serta *insight* otomatis untuk keempat halaman dashboard Streamlit:
1. **Halaman 1 — Ringkasan Eksekutif & Kalibrasi Sentimen** (`sentiment_by_app.csv`)
2. **Halaman 2 — Dinamika Bulanan & Sensitivitas Topik Boikot** (`sentiment_monthly_merged.csv` dari gabungan `sentiment_by_app_month.csv` & `blocking_topic_comparison.csv`)
3. **Halaman 3 — Audit Kualitas Versi Rilis** (`version_table.csv`)
4. **Halaman 4 — Pemodelan Tren Lintas Versi & Ekspor Data** (`version_trend.csv`)

> **Catatan:** Anda dapat menggabungkan sel pemuatan & *merge* data di awal notebook ini dengan sel audit `df_monthly` yang sudah Anda simpan sebelumnya.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results")

# 1. Load dataset utama
df_app = pd.read_csv(RESULTS_DIR / "sentiment_by_app.csv")
df_version = pd.read_csv(RESULTS_DIR / "version_table.csv")
df_trend = pd.read_csv(RESULTS_DIR / "version_trend.csv")

# 2. Load atau gabungkan data bulanan (jika sentiment_monthly_merged.csv belum ada)
merged_path = RESULTS_DIR / "sentiment_monthly_merged.csv"
if merged_path.exists():
    df_monthly = pd.read_csv(merged_path)
else:
    df_month = pd.read_csv(RESULTS_DIR / "sentiment_by_app_month.csv")
    df_blocking = pd.read_csv(RESULTS_DIR / "blocking_topic_comparison.csv")
    df_monthly = (
        pd.merge(
            df_month,
            df_blocking[["app_name", "review_month", "n_without", "negative_without", "difference"]],
            on=["app_name", "review_month"],
            how="left",
        )
        .sort_values(["app_name", "review_month"])
        .reset_index(drop=True)
    )

print("Aplikasi terdeteksi:", df_app["app_name"].unique().tolist())
print(f"Shape -> df_app: {df_app.shape}, df_monthly: {df_monthly.shape}, df_version: {df_version.shape}, df_trend: {df_trend.shape}")

Aplikasi terdeteksi: ['BRImo', 'Livin_by_Mandiri', 'myBCA', 'wondr_by_BNI']
Shape -> df_app: (4, 17), df_monthly: (32, 22), df_version: (209, 10), df_trend: (12, 8)


---
## Halaman 1: Ringkasan Eksekutif & Kalibrasi Sentimen (`sentiment_by_app.csv`)

Perhitungan yang dilakukan pada halaman ini meliputi:
* **Net Sentiment Score (NSS):** $(\text{positive\_corrected} - \text{negative\_corrected}) \times 100$ (rentang $-100$ hingga $+100$).
* **Polarization Ratio:** $\frac{\text{positive\_corrected} + \text{negative\_corrected}}{\text{neutral\_corrected}}$ (rasio opini ekstrem terhadap netral).
* **Calibration Shift:** $\text{corrected} - \text{predicted}$ untuk melihat seberapa besar koreksi prevalensi menggeser prediksi keras model.
* **Lebar 95% CI:** $\text{ci\_high} - \text{ci\_low}$ untuk mengukur presisi estimasi berdasarkan ukuran sampel `n`.
* **4 KPI Scorecards** (rata-rata tertimbang berdasarkan `n`) dan **3 Kartu Sorotan Otomatis**.

In [2]:
def compute_page1_features(df_app: pd.DataFrame, selected_apps=None, sort_by="negative_corrected"):
    df = df_app.copy()
    if selected_apps:
        df = df[df["app_name"].isin(selected_apps)].copy()

    # 1. Metrik turunan per aplikasi
    df["net_sentiment_score"] = (df["positive_corrected"] - df["negative_corrected"]) * 100
    df["polarization_ratio"] = (df["positive_corrected"] + df["negative_corrected"]) / df["neutral_corrected"].replace(0, np.nan)

    for cls in ["negative", "neutral", "positive"]:
        df[f"{cls}_shift_pred"] = df[f"{cls}_corrected"] - df[f"{cls}_predicted"]
        df[f"{cls}_shift_soft"] = df[f"{cls}_corrected"] - df[f"{cls}_soft"]
        df[f"{cls}_ci_width"] = df[f"{cls}_ci_high"] - df[f"{cls}_ci_low"]
        df[f"{cls}_est_count"] = (df[f"{cls}_corrected"] * df["n"]).round().astype(int)

    # 2. Pengurutan sesuai kontrol UI
    sort_map = {
        "negative_corrected": ("negative_corrected", False),
        "positive_corrected": ("positive_corrected", False),
        "net_sentiment_score": ("net_sentiment_score", False),
        "n": ("n", False),
    }
    col_sort, asc = sort_map.get(sort_by, ("negative_corrected", False))
    df = df.sort_values(col_sort, ascending=asc).reset_index(drop=True)

    # 3. 4 KPI Scorecards (weighted average berdasarkan n)
    total_n = int(df["n"].sum())
    weighted_neg = np.average(df["negative_corrected"], weights=df["n"])
    weighted_pos = np.average(df["positive_corrected"], weights=df["n"])
    weighted_nss = (weighted_pos - weighted_neg) * 100
    mean_abs_shift = df["negative_shift_pred"].abs().mean()

    top_vol_row = df.loc[df["n"].idxmax()]
    top_neg_row = df.loc[df["negative_corrected"].idxmax()]
    max_shift_row = df.loc[df["negative_shift_pred"].abs().idxmax()]
    narrowest_ci_row = df.loc[df["negative_ci_width"].idxmin()]
    widest_ci_row = df.loc[df["negative_ci_width"].idxmax()]

    kpis = {
        "total_ulasan": total_n,
        "sub_total_ulasan": f"{top_vol_row['app_name']} terbanyak: {top_vol_row['n']:,} ulasan",
        "tingkat_negatif_terkalibrasi": round(weighted_neg * 100, 2),
        "sub_tingkat_negatif": f"Tertinggi: {top_neg_row['app_name']} ({top_neg_row['negative_corrected']*100:.1f}%)",
        "net_sentiment_score": round(weighted_nss, 2),
        "pergeseran_bias_abs_rata2": round(mean_abs_shift * 100, 2),
    }

    # 4. 3 Kartu Sorotan Otomatis (Insight Callouts)
    insights = {
        "1_keluhan_tertinggi": (
            f"{top_neg_row['app_name']} mencatat keluhan tertinggi sebesar {top_neg_row['negative_corrected']*100:.1f}% "
            f"(95% CI: {top_neg_row['negative_ci_low']*100:.1f}%–{top_neg_row['negative_ci_high']*100:.1f}%)."
        ),
        "2_dampak_kalibrasi": (
            f"Koreksi bias terbesar terjadi pada {max_shift_row['app_name']} "
            f"({max_shift_row['negative_shift_pred']*100:+.2f} poin persentase dibanding prediksi mentah)."
        ),
        "3_presisi_estimasi": (
            f"Rentang 95% CI tersempit pada {narrowest_ci_row['app_name']} "
            f"(lebar {narrowest_ci_row['negative_ci_width']*100:.2f}%, n={narrowest_ci_row['n']:,}) dan terlebar pada "
            f"{widest_ci_row['app_name']} (lebar {widest_ci_row['negative_ci_width']*100:.2f}%, n={widest_ci_row['n']:,})."
        ),
    }

    return df, kpis, insights

df_p1, kpis_p1, insights_p1 = compute_page1_features(df_app)
print("=== KPI Halaman 1 ===")
for k, v in kpis_p1.items():
    print(f"- {k}: {v}")
print("\n=== Insights Halaman 1 ===")
for k, v in insights_p1.items():
    print(f"- {k}: {v}")
display(df_p1[["app_name", "n", "negative_predicted", "negative_soft", "negative_corrected", "negative_shift_pred", "negative_ci_width", "net_sentiment_score", "polarization_ratio"]])

=== KPI Halaman 1 ===
- total_ulasan: 134910
- sub_total_ulasan: BRImo terbanyak: 59,506 ulasan
- tingkat_negatif_terkalibrasi: 41.44
- sub_tingkat_negatif: Tertinggi: Livin_by_Mandiri (70.2%)
- net_sentiment_score: 10.82
- pergeseran_bias_abs_rata2: 0.87

=== Insights Halaman 1 ===
- 1_keluhan_tertinggi: Livin_by_Mandiri mencatat keluhan tertinggi sebesar 70.2% (95% CI: 66.8%–71.6%).
- 2_dampak_kalibrasi: Koreksi bias terbesar terjadi pada wondr_by_BNI (-1.65 poin persentase dibanding prediksi mentah).
- 3_presisi_estimasi: Rentang 95% CI tersempit pada Livin_by_Mandiri (lebar 4.78%, n=35,968) dan terlebar pada wondr_by_BNI (lebar 6.78%, n=30,604).


,app_name,n,negative_predicted,negative_soft,negative_corrected,negative_shift_pred,negative_ci_width,net_sentiment_score,polarization_ratio
0,Livin_by_Mandiri,35968,0.700512,0.694746,0.702435,0.001924,0.047808,-44.784048,22.272237
1,myBCA,8832,0.645833,0.644642,0.643773,-0.002060,0.060239,-35.379485,14.094568
2,BRImo,59506,0.303448,0.303290,0.289290,-0.014158,0.058658,35.539485,14.145952
3,wondr_by_BNI,30604,0.269507,0.269682,0.252964,-0.016544,0.067802,41.459212,11.581681


---
## Halaman 2: Dinamika Bulanan & Sensitivitas Topik Boikot (`sentiment_monthly_merged.csv`)

Perhitungan pada halaman ini mencakup:
* **Month-over-Month (MoM) Delta:** $\Delta_t = \text{negative\_corrected}_t - \text{negative\_corrected}_{t-1}$ per aplikasi.
* **Kontrol `thin_cell`:** Opsi menyembunyikan sel bersampel tipis (`thin_cell == True`) atau menandainya untuk visualisasi *hollow marker*.
* **Analisis Volatilitas & Korelasi Volume:** Rata-rata bulanan, standar deviasi ($\text{SD}$), median/IQR per aplikasi (*boxplot*), serta korelasi Pearson antara volume ulasan `n` dan `negative_corrected`.
* **Sensitivitas Topik Boikot/Blokir:** Perbandingan `negative_corrected` (semua topik) vs. `negative_without` (tanpa topik boikot) serta kolom `difference`.

In [3]:
def compute_page2_features(
    df_monthly: pd.DataFrame,
    selected_apps=None,
    month_range=None,
    hide_thin_cell=False,
    spike_threshold=0.05,
):
    df = df_monthly.sort_values(["app_name", "review_month"]).copy()

    # Hitung MoM Delta sebelum filtering rentang bulan agar bulan awal tetap akurat
    df["mom_neg_delta"] = df.groupby("app_name")["negative_corrected"].diff()
    df["mom_n_pct_change"] = df.groupby("app_name")["n"].pct_change() * 100
    df["is_spike"] = df["mom_neg_delta"] > spike_threshold

    if selected_apps:
        df = df[df["app_name"].isin(selected_apps)].copy()
    if month_range:
        start_m, end_m = month_range
        df = df[(df["review_month"] >= start_m) & (df["review_month"] <= end_m)].copy()
    if hide_thin_cell:
        df = df[~df["thin_cell"]].copy()

    df = df.reset_index(drop=True)

    # 1. Ringkasan volatilitas & korelasi volume per aplikasi (untuk Boxplot & Scatter)
    app_volatility = (
        df.groupby("app_name", as_index=False)
        .agg(
            months=("review_month", "count"),
            thin_cells=("thin_cell", "sum"),
            mean_neg=("negative_corrected", "mean"),
            std_neg=("negative_corrected", "std"),
            median_neg=("negative_corrected", "median"),
            iqr_neg=("negative_corrected", lambda s: s.quantile(0.75) - s.quantile(0.25)),
            min_neg=("negative_corrected", "min"),
            max_neg=("negative_corrected", "max"),
            mean_boycott_diff=("difference", "mean"),
            max_boycott_diff=("difference", "max"),
            corr_n_vs_neg=("n", lambda s: s.corr(df.loc[s.index, "negative_corrected"])),
        )
    )

    # 2. Ringkasan sensitivitas boikot agregat per aplikasi (untuk panel kanan Halaman 2)
    boycott_summary = (
        df.groupby("app_name", as_index=False)
        .apply(
            lambda g: pd.Series({
                "neg_with_boycott": np.average(g["negative_corrected"], weights=g["n"]),
                "neg_without_boycott": np.average(g["negative_without"].fillna(g["negative_corrected"]), weights=g["n_without"].fillna(g["n"])),
                "mean_diff": g["difference"].mean(),
                "max_diff": g["difference"].max(),
            }),
            include_groups=False,
        )
    )
    boycott_summary["weighted_diff"] = boycott_summary["neg_with_boycott"] - boycott_summary["neg_without_boycott"]

    # 3. 4 KPI Scorecards
    peak_row = df.loc[df["negative_corrected"].idxmax()]
    mean_neg_all = df["negative_corrected"].mean()
    std_neg_all = df["negative_corrected"].std()

    valid_mom = df.dropna(subset=["mom_neg_delta"])
    max_spike_row = valid_mom.loc[valid_mom["mom_neg_delta"].idxmax()] if not valid_mom.empty else peak_row
    max_diff_row = df.loc[df["difference"].idxmax()]

    kpis = {
        "puncak_keluhan_bulanan": round(peak_row["negative_corrected"] * 100, 2),
        "sub_puncak": f"{peak_row['review_month']} ({peak_row['app_name']}, n={peak_row['n']:,})",
        "rata2_dan_volatilitas": f"{mean_neg_all*100:.1f}% ± {std_neg_all*100:.1f}%",
        "lonjakan_mom_terbesar": f"{max_spike_row['mom_neg_delta']*100:+.1f}%",
        "sub_lonjakan": f"{max_spike_row['app_name']} pada {max_spike_row['review_month']}",
        "dampak_maks_boikot": f"{max_diff_row['difference']*100:+.2f}%",
        "sub_boikot": f"{max_diff_row['app_name']} ({max_diff_row['review_month']})",
    }

    # 4. 3 Kartu Sorotan Otomatis
    insights = {
        "1_lonjakan_terbesar": (
            f"{max_spike_row['app_name']} mencatat lonjakan bulanan tertinggi ({max_spike_row['mom_neg_delta']*100:+.1f} poin) "
            f"pada {max_spike_row['review_month']}."
        ),
        "2_puncak_keluhan": (
            f"Rekor keluhan bulanan mencapai {peak_row['negative_corrected']*100:.1f}% pada {peak_row['app_name']} "
            f"({peak_row['review_month']}) dengan {peak_row['n']:,} ulasan."
        ),
        "3_sensitivitas_boikot": (
            f"Rata-rata kontribusi topik boikot/blokir hanya {df['difference'].mean()*100:.2f}% "
            f"(maksimum {max_diff_row['difference']*100:.2f}% pada {max_diff_row['app_name']} bulan {max_diff_row['review_month']})."
        ),
    }

    return df, app_volatility, boycott_summary, kpis, insights

df_p2, vol_p2, boycott_p2, kpis_p2, insights_p2 = compute_page2_features(df_monthly, hide_thin_cell=False)
print("=== KPI Halaman 2 ===")
for k, v in kpis_p2.items():
    print(f"- {k}: {v}")
print("\n=== Insights Halaman 2 ===")
for k, v in insights_p2.items():
    print(f"- {k}: {v}")
display(vol_p2)
display(boycott_p2)

=== KPI Halaman 2 ===
- puncak_keluhan_bulanan: 91.82
- sub_puncak: 2026-08 (Livin_by_Mandiri, n=18,463)
- rata2_dan_volatilitas: 43.7% ± 19.6%
- lonjakan_mom_terbesar: +60.4%
- sub_lonjakan: Livin_by_Mandiri pada 2026-08
- dampak_maks_boikot: +3.55%
- sub_boikot: Livin_by_Mandiri (2026-08)

=== Insights Halaman 2 ===
- 1_lonjakan_terbesar: Livin_by_Mandiri mencatat lonjakan bulanan tertinggi (+60.4 poin) pada 2026-08.
- 2_puncak_keluhan: Rekor keluhan bulanan mencapai 91.8% pada Livin_by_Mandiri (2026-08) dengan 18,463 ulasan.
- 3_sensitivitas_boikot: Rata-rata kontribusi topik boikot/blokir hanya 0.15% (maksimum 3.55% pada Livin_by_Mandiri bulan 2026-08).


,app_name,months,thin_cells,mean_neg,std_neg,median_neg,iqr_neg,min_neg,max_neg,mean_boycott_diff,max_boycott_diff,corr_n_vs_neg
0,BRImo,8,0,0.292183,0.043323,0.290881,0.033300,0.208397,0.346940,0.000295,0.000992,-0.424734
1,Livin_by_Mandiri,8,0,0.572527,0.184218,0.564803,0.129833,0.314428,0.918174,0.004785,0.035484,0.624902
2,myBCA,8,0,0.630994,0.071218,0.616935,0.120571,0.546813,0.719836,0.000564,0.001167,0.671204
3,wondr_by_BNI,8,0,0.251923,0.038856,0.244512,0.031972,0.196692,0.321260,0.000349,0.000841,0.367448


,app_name,neg_with_boycott,neg_without_boycott,mean_diff,max_diff,weighted_diff
0,BRImo,0.289290,0.289021,0.000295,0.000992,0.000269
1,Livin_by_Mandiri,0.702435,0.650532,0.004785,0.035484,0.051903
2,myBCA,0.643773,0.643298,0.000564,0.001167,0.000475
3,wondr_by_BNI,0.252964,0.252614,0.000349,0.000841,0.000350


---
## Halaman 3: Audit Kualitas Versi Rilis (`version_table.csv`)

Perhitungan pada halaman ini mencakup:
* **Durasi Masa Aktif Versi (Hari):** $(\text{last\_date} - \text{first\_date}).\text{days} + 1$.
* **Audit Kalibrasi Versi:** $\text{negative\_share} - \text{soft\_negative}$ (mendeteksi versi dengan prediksi probabilistik yang ambigu).
* **Lebar 95% CI Versi:** $\text{ci\_high} - \text{ci\_low}$.
* **Kurva Konsentrasi Pareto:** Persentase kumulatif ulasan yang terkonsentrasi pada urutan versi teratas per aplikasi.
* **Leaderboard Top-N Versi Terburuk & Terbaik** (berdasarkan versi yang memenuhi kriteria reliabilitas).

In [4]:
def compute_page3_features(
    df_version: pd.DataFrame,
    selected_apps=None,
    only_reliable=True,
    min_reviews=1,
    search_version="",
    top_n=5,
):
    df_raw = df_version.copy()
    if selected_apps:
        df_raw = df_raw[df_raw["app_name"].isin(selected_apps)].copy()

    # Konversi tanggal & hitung metrik turunan siklus hidup versi
    df_raw["first_date"] = pd.to_datetime(df_raw["first_date"])
    df_raw["last_date"] = pd.to_datetime(df_raw["last_date"])
    df_raw["active_days"] = (df_raw["last_date"] - df_raw["first_date"]).dt.days + 1
    df_raw["soft_hard_diff"] = df_raw["negative_share"] - df_raw["soft_negative"]
    df_raw["ci_width"] = df_raw["ci_high"] - df_raw["ci_low"]

    # Hitung cakupan ulasan reliabel sebelum filter diterapkan
    total_versions_all = len(df_raw)
    reliable_versions_all = int(df_raw["reliable"].sum())
    total_reviews_all = int(df_raw["n"].sum())
    reliable_reviews_all = int(df_raw.loc[df_raw["reliable"], "n"].sum())
    reliable_coverage_pct = (reliable_reviews_all / total_reviews_all * 100) if total_reviews_all > 0 else 0.0

    # Terapkan filter UI
    df_filtered = df_raw.copy()
    if only_reliable:
        df_filtered = df_filtered[df_filtered["reliable"]].copy()
    if min_reviews > 1:
        df_filtered = df_filtered[df_filtered["n"] >= min_reviews].copy()
    if search_version.strip():
        df_filtered = df_filtered[
            df_filtered["app_version"].astype(str).str.contains(search_version.strip(), case=False, na=False)
        ].copy()

    df_filtered = df_filtered.sort_values(["app_name", "first_date", "app_version"]).reset_index(drop=True)

    # Kurva Pareto konsentrasi ulasan per aplikasi
    df_pareto = df_raw.sort_values(["app_name", "n"], ascending=[True, False]).copy()
    df_pareto["cum_reviews"] = df_pareto.groupby("app_name")["n"].cumsum()
    df_pareto["total_app_reviews"] = df_pareto.groupby("app_name")["n"].transform("sum")
    df_pareto["cum_review_share"] = df_pareto["cum_reviews"] / df_pareto["total_app_reviews"] * 100
    df_pareto["version_rank"] = df_pareto.groupby("app_name").cumcount() + 1

    # Leaderboard Top-N versi terburuk & terbaik dari data terfilter
    worst_versions = df_filtered.sort_values(["negative_share", "n"], ascending=[False, False]).head(top_n)
    best_versions = df_filtered.sort_values(["negative_share", "n"], ascending=[True, False]).head(top_n)

    # 4 KPI Scorecards
    median_days = df_filtered["active_days"].median() if not df_filtered.empty else 0
    longest_app = (
        df_filtered.groupby("app_name")["active_days"].median().idxmax()
        if not df_filtered.empty else "-"
    )
    worst_row = df_filtered.loc[df_filtered["negative_share"].idxmax()] if not df_filtered.empty else None
    max_soft_diff_row = df_filtered.loc[df_filtered["soft_hard_diff"].abs().idxmax()] if not df_filtered.empty else None

    kpis = {
        "total_versi_terdeteksi": f"{total_versions_all} versi ({reliable_versions_all} reliabel)",
        "cakupan_ulasan_reliabel": f"{reliable_coverage_pct:.1f}%",
        "sub_cakupan": f"{reliable_reviews_all:,} dari {total_reviews_all:,} ulasan berversi",
        "median_masa_aktif": f"{median_days:.0f} Hari",
        "sub_masa_aktif": f"Median terlama: {longest_app}",
        "versi_keluhan_tertinggi": f"{worst_row['negative_share']*100:.1f}%" if worst_row is not None else "-",
        "sub_versi_tertinggi": f"{worst_row['app_name']} v{worst_row['app_version']} (n={worst_row['n']:,})" if worst_row is not None else "-",
    }

    # 3 Kartu Sorotan Otomatis
    top5_share = df_pareto[df_pareto["version_rank"] <= 5].groupby("app_name")["cum_review_share"].max().mean()
    insights = {
        "1_konsentrasi_rilis": (
            f"Sebanyak {reliable_coverage_pct:.1f}% ulasan terkumpul pada {reliable_versions_all} versi reliabel; "
            f"rata-rata 5 versi teratas mencakup {top5_share:.1f}% total ulasan tiap bank."
        ),
        "2_anomali_rilis": (
            f"Keluhan tertinggi pada sampel terpilih tercatat pada {worst_row['app_name']} versi "
            f"{worst_row['app_version']} sebesar {worst_row['negative_share']*100:.1f}% (n={worst_row['n']:,})."
            if worst_row is not None else "Tidak ada versi yang memenuhi filter."
        ),
        "3_audit_kalibrasi_versi": (
            f"Selisih terbesar antara prediksi keras dan probabilitas softmax muncul pada "
            f"{max_soft_diff_row['app_name']} v{max_soft_diff_row['app_version']} ({max_soft_diff_row['soft_hard_diff']*100:+.2f}%)."
            if max_soft_diff_row is not None else "-"
        ),
    }

    return df_filtered, df_pareto, worst_versions, best_versions, kpis, insights

df_p3, pareto_p3, worst_p3, best_p3, kpis_p3, insights_p3 = compute_page3_features(df_version, only_reliable=True)
print("=== KPI Halaman 3 ===")
for k, v in kpis_p3.items():
    print(f"- {k}: {v}")
print("\n=== Insights Halaman 3 ===")
for k, v in insights_p3.items():
    print(f"- {k}: {v}")
display(worst_p3[["app_name", "app_version", "n", "active_days", "negative_share", "ci_low", "ci_high", "soft_negative"]])

=== KPI Halaman 3 ===
- total_versi_terdeteksi: 209 versi (42 reliabel)
- cakupan_ulasan_reliabel: 98.1%
- sub_cakupan: 104,082 dari 106,109 ulasan berversi
- median_masa_aktif: 156 Hari
- sub_masa_aktif: Median terlama: Livin_by_Mandiri
- versi_keluhan_tertinggi: 93.0%
- sub_versi_tertinggi: Livin_by_Mandiri v1.4.0 (n=100)

=== Insights Halaman 3 ===
- 1_konsentrasi_rilis: Sebanyak 98.1% ulasan terkumpul pada 42 versi reliabel; rata-rata 5 versi teratas mencakup 79.7% total ulasan tiap bank.
- 2_anomali_rilis: Keluhan tertinggi pada sampel terpilih tercatat pada Livin_by_Mandiri versi 1.4.0 sebesar 93.0% (n=100).
- 3_audit_kalibrasi_versi: Selisih terbesar antara prediksi keras dan probabilitas softmax muncul pada Livin_by_Mandiri v3.0.0 (+1.55%).


,app_name,app_version,n,active_days,negative_share,ci_low,ci_high,soft_negative
22,Livin_by_Mandiri,1.4.0,100,223,0.930000,0.862503,0.965681,0.919847
21,Livin_by_Mandiri,1.9.0,154,237,0.889610,0.830328,0.929928,0.880293
16,Livin_by_Mandiri,2.1.0,142,240,0.866197,0.800462,0.912640,0.861556
26,Livin_by_Mandiri,3.7.0,7895,48,0.851425,0.843408,0.859100,0.843383
19,Livin_by_Mandiri,2.2.0,173,242,0.820809,0.756887,0.870793,0.809923


---
## Halaman 4: Pemodelan Tren Lintas Versi & Matriks Komparasi (`version_trend.csv`)

Perhitungan pada halaman ini mencakup:
* **Efek Persentase per Kenaikan Versi:** $(\text{odds\_ratio\_per\_version} - 1) \times 100$.
* **Status Signifikansi Statistik (95% CI vs. 1,0):**
  * `Signifikan Membaik (OR < 1)` jika `or_ci_high < 1.0`
  * `Signifikan Memburuk (OR > 1)` jika `or_ci_low > 1.0`
  * `Tidak Signifikan` jika interval melewati `1.0`
* **Uji Ketahanan (*Robustness Delta*):** Selisih *Odds Ratio* antara skenario `all versioned reviews` vs. `without blocking/boycott topic`.
* **Summary Benchmark Table:** Penggabungan ringkasan metrik dari Halaman 1, 2, 3, dan 4 ke dalam satu matriks pembanding lintas 4 aplikasi.

In [5]:
def classify_or_significance(row):
    if row["or_ci_high"] < 1.0:
        return "Signifikan Membaik (OR < 1)"
    elif row["or_ci_low"] > 1.0:
        return "Signifikan Memburuk (OR > 1)"
    return "Tidak Signifikan"

def compute_page4_features(
    df_trend: pd.DataFrame,
    df_app: pd.DataFrame,
    df_monthly: pd.DataFrame,
    df_version: pd.DataFrame,
    selected_apps=None,
    selected_subsets=None,
):
    df = df_trend.copy()
    if selected_apps:
        df = df[df["app_name"].isin(selected_apps)].copy()
    if selected_subsets:
        df = df[df["subset"].isin(selected_subsets)].copy()

    # 1. Metrik turunan regresi
    df["pct_odds_change_per_version"] = (df["odds_ratio_per_version"] - 1.0) * 100
    df["or_ci_width"] = df["or_ci_high"] - df["or_ci_low"]
    df["significance_status"] = df.apply(classify_or_significance, axis=1)

    # 2. Audit Robustness per aplikasi (all vs without blocking/boycott topic)
    piv_or = df_trend.pivot(index="app_name", columns="subset", values="odds_ratio_per_version")
    piv_sig = (
        df_trend.assign(sig=df_trend.apply(classify_or_significance, axis=1))
        .pivot(index="app_name", columns="subset", values="sig")
    )
    robustness_df = pd.DataFrame({
        "app_name": piv_or.index,
        "or_all": piv_or.get("all versioned reviews"),
        "or_reliable": piv_or.get("reliable versions only"),
        "or_without_boycott": piv_or.get("without blocking/boycott topic"),
    }).reset_index(drop=True)
    robustness_df["boycott_or_shift"] = robustness_df["or_all"] - robustness_df["or_without_boycott"]
    robustness_df["is_direction_consistent"] = (
        (piv_sig.get("all versioned reviews") == piv_sig.get("without blocking/boycott topic"))
        & (piv_sig.get("all versioned reviews") == piv_sig.get("reliable versions only"))
    ).values

    if selected_apps:
        robustness_df = robustness_df[robustness_df["app_name"].isin(selected_apps)].reset_index(drop=True)

    # 3. 4 KPI Scorecards
    reliable_sub = df[df["subset"] == "reliable versions only"]
    ref_df = reliable_sub if not reliable_sub.empty else df

    best_or_row = ref_df.loc[ref_df["odds_ratio_per_version"].idxmin()]
    mean_corr_chrono = ref_df["corr_version_month"].mean()
    consistent_count = int(robustness_df["is_direction_consistent"].sum())
    total_apps_count = len(robustness_df)

    all_sub = df[df["subset"] == "all versioned reviews"]
    sample_ref = all_sub if not all_sub.empty else ref_df
    total_reviews_tested = int(sample_ref["reviews"].sum())
    total_versions_tested = int(sample_ref["versions"].sum())

    kpis = {
        "perbaikan_versi_terkuat": f"OR {best_or_row['odds_ratio_per_version']:.3f} ({best_or_row['pct_odds_change_per_version']:+.2f}%/versi)",
        "sub_perbaikan": f"{best_or_row['app_name']} ({best_or_row['subset']})",
        "rata2_korelasi_kronologis": round(mean_corr_chrono, 3),
        "konsistensi_robustness": f"{consistent_count} dari {total_apps_count} Aplikasi",
        "total_sampel_teruji": f"{total_reviews_tested:,} ulasan",
        "sub_sampel_teruji": f"Mencakup {total_versions_tested} versi ({sample_ref['subset'].iloc[0]})",
    }

    # 4. 3 Kartu Sorotan Otomatis
    improving_apps = ref_df.loc[ref_df["significance_status"] == "Signifikan Membaik (OR < 1)", "app_name"].tolist()
    max_boycott_shift = robustness_df.loc[robustness_df["boycott_or_shift"].abs().idxmax()]
    top_corr_row = ref_df.loc[ref_df["corr_version_month"].idxmax()]

    insights = {
        "1_arah_tren_utama": (
            f"Aplikasi dengan tren perbaikan signifikan per versi: {', '.join(improving_apps) if improving_apps else 'Tidak ada'}. "
            f"Terkuat pada {best_or_row['app_name']} (OR = {best_or_row['odds_ratio_per_version']:.3f})."
        ),
        "2_ketahanan_boikot": (
            f"Pergeseran OR terbesar saat topik boikot dihapus terjadi pada {max_boycott_shift['app_name']} "
            f"({max_boycott_shift['boycott_or_shift']:+.5f}), menunjukkan hasil pemodelan sangat stabil."
        ),
        "3_validitas_urutan_rilis": (
            f"Korelasi kronologis tertinggi antara versi dan bulan ulasan dicapai {top_corr_row['app_name']} "
            f"(r = {top_corr_row['corr_version_month']:.3f})."
        ),
    }

    # 5. Summary Benchmark Matrix Lintas 4 Halaman
    m_agg = df_monthly.groupby("app_name", as_index=False).agg(
        monthly_neg_mean=("negative_corrected", "mean"),
        monthly_neg_peak=("negative_corrected", "max"),
        boycott_diff_max=("difference", "max"),
    )
    v_agg = df_version.groupby("app_name", as_index=False).agg(
        total_versions=("app_version", "count"),
        reliable_versions=("reliable", "sum"),
    )
    t_agg = df_trend[df_trend["subset"] == "reliable versions only"][
        ["app_name", "corr_version_month", "odds_ratio_per_version", "or_ci_low", "or_ci_high"]
    ]

    benchmark_matrix = (
        df_app[["app_name", "n", "negative_corrected", "neutral_corrected", "positive_corrected"]]
        .assign(net_sentiment_score=lambda d: (d["positive_corrected"] - d["negative_corrected"]) * 100)
        .merge(m_agg, on="app_name", how="left")
        .merge(v_agg, on="app_name", how="left")
        .merge(t_agg, on="app_name", how="left")
    )

    return df, robustness_df, benchmark_matrix, kpis, insights

df_p4, robust_p4, bench_p4, kpis_p4, insights_p4 = compute_page4_features(
    df_trend, df_app, df_monthly, df_version
)
print("=== KPI Halaman 4 ===")
for k, v in kpis_p4.items():
    print(f"- {k}: {v}")
print("\n=== Insights Halaman 4 ===")
for k, v in insights_p4.items():
    print(f"- {k}: {v}")
display(df_p4[["app_name", "subset", "reviews", "versions", "corr_version_month", "odds_ratio_per_version", "pct_odds_change_per_version", "significance_status"]])
display(bench_p4)

=== KPI Halaman 4 ===
- perbaikan_versi_terkuat: OR 0.975 (-2.47%/versi)
- sub_perbaikan: BRImo (reliable versions only)
- rata2_korelasi_kronologis: 0.82
- konsistensi_robustness: 1 dari 4 Aplikasi
- total_sampel_teruji: 106,109 ulasan
- sub_sampel_teruji: Mencakup 209 versi (all versioned reviews)

=== Insights Halaman 4 ===
- 1_arah_tren_utama: Aplikasi dengan tren perbaikan signifikan per versi: BRImo. Terkuat pada BRImo (OR = 0.975).
- 2_ketahanan_boikot: Pergeseran OR terbesar saat topik boikot dihapus terjadi pada Livin_by_Mandiri (-0.00219), menunjukkan hasil pemodelan sangat stabil.
- 3_validitas_urutan_rilis: Korelasi kronologis tertinggi antara versi dan bulan ulasan dicapai myBCA (r = 0.940).


,app_name,subset,reviews,versions,corr_version_month,odds_ratio_per_version,pct_odds_change_per_version,significance_status
0,BRImo,all versioned reviews,48586,107,0.466367,0.972196,-2.780397,Signifikan Membaik (OR < 1)
1,BRImo,reliable versions only,47783,16,0.905555,0.975300,-2.470029,Signifikan Membaik (OR < 1)
2,BRImo,without blocking/boycott topic,48569,107,0.466275,0.972205,-2.779489,Signifikan Membaik (OR < 1)
3,Livin_by_Mandiri,all versioned reviews,23107,28,0.231501,0.987375,-1.262472,Signifikan Membaik (OR < 1)
4,Livin_by_Mandiri,reliable versions only,22304,11,0.541565,1.027883,2.788265,Signifikan Memburuk (OR > 1)
5,Livin_by_Mandiri,without blocking/boycott topic,20553,28,0.267913,0.989562,-1.043790,Tidak Signifikan
6,myBCA,all versioned reviews,6883,59,0.567751,1.006510,0.650990,Tidak Signifikan
7,myBCA,reliable versions only,6579,11,0.940082,1.152424,15.242449,Signifikan Memburuk (OR > 1)
8,myBCA,without blocking/boycott topic,6876,59,0.567337,1.006474,0.647386,Tidak Signifikan
9,wondr_by_BNI,all versioned reviews,27533,15,0.831455,1.006478,0.647835,Tidak Signifikan


,app_name,n,negative_corrected,neutral_corrected,positive_corrected,net_sentiment_score,monthly_neg_mean,monthly_neg_peak,boycott_diff_max,total_versions,reliable_versions,corr_version_month,odds_ratio_per_version,or_ci_low,or_ci_high
0,BRImo,59506,0.289290,0.066024,0.644685,35.539485,0.292183,0.346940,0.000992,107,16,0.905555,0.975300,0.961231,0.989574
1,Livin_by_Mandiri,35968,0.702435,0.042970,0.254595,-44.784048,0.572527,0.918174,0.035484,28,11,0.541565,1.027883,1.009045,1.047072
2,myBCA,8832,0.643773,0.066249,0.289978,-35.379485,0.630994,0.719836,0.001167,59,11,0.940082,1.152424,1.098429,1.209074
3,wondr_by_BNI,30604,0.252964,0.079481,0.667556,41.459212,0.251923,0.321260,0.000841,15,4,0.893738,1.291295,1.199882,1.389674
